# Cohort retention analysis: SaaS usage analytics

This notebook analyzes user retention by signup cohort, plan tier, and engagement level to understand lifecycle patterns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

df = pd.read_csv("../data/saas_usage.csv")
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["last_active_date"] = pd.to_datetime(df["last_active_date"])
df["signup_month"] = df["signup_date"].dt.to_period("M")
df["last_active_month"] = df["last_active_date"].dt.to_period("M")

print(f"Dataset: {len(df)} users")
print(f"Signup range: {df['signup_date'].min().date()} to {df['signup_date'].max().date()}")
print(f"Unique signup cohorts: {df['signup_month'].nunique()}")

## Cohort retention heatmap

In [ ]:
# Calculate months between signup and last active
df["months_active"] = (
    df["last_active_month"].apply(lambda x: x.ordinal) -
    df["signup_month"].apply(lambda x: x.ordinal)
)

cohort_sizes = df.groupby("signup_month")["user_id"].nunique()

retention_data = df.groupby(["signup_month", "months_active"])["user_id"].nunique().reset_index()
retention_data.columns = ["signup_month", "months_active", "users"]

retention_pivot = retention_data.pivot(index="signup_month", columns="months_active", values="users").fillna(0)

# Normalize by cohort size
for col in retention_pivot.columns:
    retention_pivot[col] = retention_pivot[col] / cohort_sizes

# Display last 12 cohorts, months 0-12
display_cols = [c for c in retention_pivot.columns if 0 <= c <= 12]
retention_display = retention_pivot[display_cols].tail(12)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    retention_display, annot=True, fmt=".0%", cmap="YlGnBu",
    linewidths=0.5, ax=ax
)
ax.set_title("Cohort retention heatmap (last 12 cohorts)")
ax.set_xlabel("Months since signup")
ax.set_ylabel("Signup cohort")
plt.tight_layout()
plt.show()

## Retention by plan tier

In [ ]:
retention_by_plan = df.groupby("plan_tier").agg(
    total_users=("user_id", "count"),
    active_users=("is_churned", lambda x: (x == 0).sum()),
    churned_users=("is_churned", "sum"),
    avg_active_days=("monthly_active_days", "mean"),
    avg_logins=("daily_logins", "mean"),
    avg_features=("features_used", "mean"),
    avg_nps=("nps_score", "mean"),
).round(2)
retention_by_plan["retention_rate"] = (retention_by_plan["active_users"] / retention_by_plan["total_users"]).round(3)

print("Retention metrics by plan tier:")
print(retention_by_plan.to_string())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {"Free": "#EF553B", "Pro": "#636EFA", "Enterprise": "#00CC96"}
plan_order = ["Free", "Pro", "Enterprise"]

# Retention rate
retention_by_plan.loc[plan_order, "retention_rate"].plot(
    kind="bar", ax=axes[0], color=[colors[p] for p in plan_order], edgecolor="black"
)
axes[0].set_title("Retention rate by plan tier")
axes[0].set_ylabel("Retention rate")
axes[0].set_ylim(0, 1)
for i, v in enumerate(retention_by_plan.loc[plan_order, "retention_rate"]):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold")

# Average features used
retention_by_plan.loc[plan_order, "avg_features"].plot(
    kind="bar", ax=axes[1], color=[colors[p] for p in plan_order], edgecolor="black"
)
axes[1].set_title("Avg features used by plan tier")
axes[1].set_ylabel("Features used")

plt.tight_layout()
plt.show()

## Signup trend and cohort sizes

In [ ]:
# Signups per month
signups = df.groupby("signup_month")["user_id"].count()

fig, ax = plt.subplots(figsize=(14, 5))
signups.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("New signups per month")
ax.set_xlabel("Signup month")
ax.set_ylabel("New users")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Summary

Key takeaways from the cohort retention analysis:

1. **Retention is strongest for Enterprise users** -- they have deeper product integration and higher switching costs
2. **Free-tier cohorts show the steepest drop-off** in the first 1-3 months after signup
3. **Feature adoption is the key differentiator** between retained and churned users across all cohorts
4. **Early engagement matters** -- users who adopt 6+ features in their first month are significantly more likely to be retained
5. **Cohort sizes are relatively stable** -- growth is consistent but retention improvements would have the largest revenue impact